In [41]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
torch.manual_seed(42)
from sklearn.metrics import f1_score

In [2]:
def read_data(filepath):
    sentences = [[]]
    for line in open(filepath):
        line = line.strip()
        if line == "":
            sentences.append([])
        else:
            try:
                idx, word, tag = line.split()
            except ValueError:
                idx, word = line.split()
                tag = "O"
            sentences[-1].append((word, tag))
    if sentences[-1] == []:
        sentences.pop()
    return sentences

In [3]:
train_sentences = read_data(r"/home/omghag/CSCI-544-Assignment/HW3/data/train")

In [4]:
print(f"Number of sentences: {len(train_sentences)}")
print(f"First sentence: {train_sentences[0]}")
print(f"Last sentence: {train_sentences[-1]}")

Number of sentences: 14987
First sentence: [('EU', 'B-ORG'), ('rejects', 'O'), ('German', 'B-MISC'), ('call', 'O'), ('to', 'O'), ('boycott', 'O'), ('British', 'B-MISC'), ('lamb', 'O'), ('.', 'O')]
Last sentence: [('-DOCSTART-', 'O')]


In [5]:
train_sentences = [s for s in train_sentences if s[0][0] != '-DOCSTART-']

In [6]:
print(f"Number of sentences: {len(train_sentences)}")
print(f"First sentence: {train_sentences[0]}")
print(f"Last sentence: {train_sentences[-1]}")

Number of sentences: 14041
First sentence: [('EU', 'B-ORG'), ('rejects', 'O'), ('German', 'B-MISC'), ('call', 'O'), ('to', 'O'), ('boycott', 'O'), ('British', 'B-MISC'), ('lamb', 'O'), ('.', 'O')]
Last sentence: [('Swansea', 'B-ORG'), ('1', 'O'), ('Lincoln', 'B-ORG'), ('2', 'O')]


In [7]:
test_sentences = read_data(r"/home/omghag/CSCI-544-Assignment/HW3/data/test")
print(f"Number of sentences: {len(test_sentences)}")
print(f"First sentence: {test_sentences[0]}")
print(f"Last sentence: {test_sentences[-1]}")

Number of sentences: 3684
First sentence: [('SOCCER', 'O'), ('-', 'O'), ('JAPAN', 'O'), ('GET', 'O'), ('LUCKY', 'O'), ('WIN', 'O'), (',', 'O'), ('CHINA', 'O'), ('IN', 'O'), ('SURPRISE', 'O'), ('DEFEAT', 'O'), ('.', 'O')]
Last sentence: [('-DOCSTART-', 'O')]


In [8]:
test_sentences = [s for s in test_sentences if s[0][0] != '-DOCSTART-']

In [9]:
print(f"Number of sentences: {len(test_sentences)}")
print(f"First sentence: {test_sentences[0]}")
print(f"Last sentence: {test_sentences[-1]}")

Number of sentences: 3453
First sentence: [('SOCCER', 'O'), ('-', 'O'), ('JAPAN', 'O'), ('GET', 'O'), ('LUCKY', 'O'), ('WIN', 'O'), (',', 'O'), ('CHINA', 'O'), ('IN', 'O'), ('SURPRISE', 'O'), ('DEFEAT', 'O'), ('.', 'O')]
Last sentence: [('The', 'O'), ('lanky', 'O'), ('former', 'O'), ('Leeds', 'O'), ('United', 'O'), ('defender', 'O'), ('did', 'O'), ('not', 'O'), ('make', 'O'), ('his', 'O'), ('England', 'O'), ('debut', 'O'), ('until', 'O'), ('the', 'O'), ('age', 'O'), ('of', 'O'), ('30', 'O'), ('but', 'O'), ('eventually', 'O'), ('won', 'O'), ('35', 'O'), ('caps', 'O'), ('and', 'O'), ('was', 'O'), ('a', 'O'), ('key', 'O'), ('member', 'O'), ('of', 'O'), ('the', 'O'), ('1966', 'O'), ('World', 'O'), ('Cup', 'O'), ('winning', 'O'), ('team', 'O'), ('with', 'O'), ('his', 'O'), ('younger', 'O'), ('brother', 'O'), (',', 'O'), ('Bobby', 'O'), ('.', 'O')]


In [10]:
dev_sentences = read_data(r"/home/omghag/CSCI-544-Assignment/HW3/data/dev")
print(f"Number of sentences: {len(dev_sentences)}")
print(f"First sentence: {dev_sentences[0]}")
print(f"Last sentence: {dev_sentences[-1]}")

Number of sentences: 3466
First sentence: [('CRICKET', 'O'), ('-', 'O'), ('LEICESTERSHIRE', 'B-ORG'), ('TAKE', 'O'), ('OVER', 'O'), ('AT', 'O'), ('TOP', 'O'), ('AFTER', 'O'), ('INNINGS', 'O'), ('VICTORY', 'O'), ('.', 'O')]
Last sentence: [('-DOCSTART-', 'O')]


In [11]:
dev_sentences = [s for s in dev_sentences if s[0][0] != '-DOCSTART-']
print(f"Number of sentences: {len(dev_sentences)}")
print(f"First sentence: {dev_sentences[0]}")
print(f"Last sentence: {dev_sentences[-1]}")

Number of sentences: 3250
First sentence: [('CRICKET', 'O'), ('-', 'O'), ('LEICESTERSHIRE', 'B-ORG'), ('TAKE', 'O'), ('OVER', 'O'), ('AT', 'O'), ('TOP', 'O'), ('AFTER', 'O'), ('INNINGS', 'O'), ('VICTORY', 'O'), ('.', 'O')]
Last sentence: [('--', 'O'), ('Dhaka', 'B-ORG'), ('Newsroom', 'I-ORG'), ('880-2-506363', 'O')]


In [12]:
def build_vocab(sentences):
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    tag2idx = {"<PAD>": 0}
    
    for sentence in sentences:
        for word, tag in sentence:
            if word not in word2idx:
                word2idx[word] = len(word2idx)
            if tag not in tag2idx:
                tag2idx[tag] = len(tag2idx)
    
    return word2idx, tag2idx

In [13]:
word2idx, tag2idx = build_vocab(train_sentences)
print(f"Number of unique words: {len(word2idx)}")
print(f"Number of unique tags: {len(tag2idx)}")
tag2idx

Number of unique words: 23625
Number of unique tags: 10


{'<PAD>': 0,
 'B-ORG': 1,
 'O': 2,
 'B-MISC': 3,
 'B-PER': 4,
 'I-PER': 5,
 'B-LOC': 6,
 'I-ORG': 7,
 'I-MISC': 8,
 'I-LOC': 9}

In [14]:
idx2tag = {v: k for k, v in tag2idx.items()}

In [15]:
def encode_data(sentences, word2idx, tag2idx):
    word_ids = [[word2idx.get(word, word2idx["<UNK>"]) for word, tag in sentence] for sentence in sentences]
    tag_ids = [[tag2idx[tag] for word, tag in sentence] for sentence in sentences]
    return word_ids, tag_ids

In [16]:
train_words, train_tags = encode_data(train_sentences, word2idx, tag2idx)
dev_words, dev_tags = encode_data(dev_sentences, word2idx, tag2idx)
test_words, test_tags = encode_data(test_sentences, word2idx, tag2idx)

In [18]:
class NERDataset(Dataset):
    def __init__(self, word_ids, tag_ids):
        self.word_ids = word_ids
        self.tag_ids = tag_ids

    def __len__(self):
        return len(self.word_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.word_ids[idx]), torch.tensor(self.tag_ids[idx])

In [19]:
def collate_fn(batch):
    word_ids, tag_ids = zip(*batch)
    word_ids_padded = pad_sequence(word_ids, batch_first=True, padding_value=0)
    tag_ids_padded = pad_sequence(tag_ids, batch_first=True, padding_value=0)
    return word_ids_padded, tag_ids_padded

In [21]:
train_dataset = NERDataset(train_words, train_tags)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [169]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(NERDataset(dev_words, dev_tags), batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(NERDataset(test_words, test_tags), batch_size=64, shuffle=False, collate_fn=collate_fn)

In [170]:
import torch.nn as nn

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, 
                 dropout, linear_dim, num_tags):
        super(BLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.blstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                            dropout=dropout, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden_dim * 2, linear_dim)
        self.elu = nn.ELU()
        self.classifier = nn.Linear(linear_dim, num_tags)
    
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.blstm(x)
        x = self.linear(x)
        x = self.elu(x)
        x = self.classifier(x)
        return x


In [171]:
model = BLSTM(
    vocab_size=len(word2idx),
    embedding_dim=100,
    hidden_dim=256,
    num_layers=1,
    dropout=0.33,
    linear_dim=128,
    num_tags=len(tag2idx)
)

In [172]:
words, tags = next(iter(train_loader))
print(f"Input shape: {words.shape}")
output = model(words)
print(f"Output shape: {output.shape}")

Input shape: torch.Size([64, 48])
Output shape: torch.Size([64, 48, 10])


In [173]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model= model.to(device)

In [174]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for words, tags in loader:
        words = words.to(device)
        tags = tags.to(device)
        # 1. zero gradients
        optimizer.zero_grad()
        # 2. forward pass
        output = model(words)
        # 3. reshape output and tags
        output = output.view(-1, output.size(-1))
        tags = tags.view(-1)
        # 4. compute loss
        loss = criterion(output, tags)
        # 5. backward pass
        loss.backward()
        # 6. update weights
        optimizer.step()
        # 7. accumulate loss
        total_loss += loss.item()
    return total_loss / len(loader)

In [175]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_tags = []
    
    with torch.no_grad():
        for words, tags in loader:
            words = words.to(device)
            tags = tags.to(device)
            # forward pass + loss
            output = model(words)
            loss = criterion(output.view(-1, output.size(-1)), tags.view(-1))
            total_loss += loss.item()

            # collect predictions using argmax
            preds = torch.argmax(output, dim=-1)
            
            for pred_seq, tag_seq in zip(preds, tags):
                for p, t in zip(pred_seq, tag_seq):
                    if t != 0:  # ignore padding
                        all_preds.append(p.item())
                        all_tags.append(t.item())


    return total_loss / len(loader), all_preds, all_tags

In [176]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [177]:
best_val_loss = float('inf')
patience_counter = 0
patience = 5
num_epochs = 50
for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, preds, true_tags = evaluate(model, dev_loader, criterion)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'blstm1.pt')
        patience_counter = 0
        print(f"  ✓ Best model saved!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print("Early stopping!")
            break

model.load_state_dict(torch.load('blstm1.pt'))

Epoch 1/50 | Train Loss: 0.7273 | Val Loss: 0.6080
  ✓ Best model saved!
Epoch 2/50 | Train Loss: 0.4889 | Val Loss: 0.5323
  ✓ Best model saved!
Epoch 3/50 | Train Loss: 0.3663 | Val Loss: 0.4514
  ✓ Best model saved!
Epoch 4/50 | Train Loss: 0.2799 | Val Loss: 0.4712
  No improvement (1/5)
Epoch 5/50 | Train Loss: 0.2084 | Val Loss: 0.3728
  ✓ Best model saved!
Epoch 6/50 | Train Loss: 0.1541 | Val Loss: 0.3988
  No improvement (1/5)
Epoch 7/50 | Train Loss: 0.1121 | Val Loss: 0.3147
  ✓ Best model saved!
Epoch 8/50 | Train Loss: 0.0826 | Val Loss: 0.4990
  No improvement (1/5)
Epoch 9/50 | Train Loss: 0.0577 | Val Loss: 0.4471
  No improvement (2/5)
Epoch 10/50 | Train Loss: 0.0393 | Val Loss: 0.4313
  No improvement (3/5)
Epoch 11/50 | Train Loss: 0.0256 | Val Loss: 0.5102
  No improvement (4/5)
Epoch 12/50 | Train Loss: 0.0155 | Val Loss: 0.5164
  No improvement (5/5)
Early stopping!


<All keys matched successfully>

In [178]:
loss, preds, true_tags = evaluate(model, dev_loader, criterion)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]

Loss: 0.3147


In [179]:
# how many O tags vs B-PER, I-LOC etc in your dataset?
from collections import Counter
all_train_tags = [tag for sent in train_sentences for _, tag in sent]
print(Counter(all_train_tags))

Counter({'O': 169578, 'B-LOC': 7140, 'B-PER': 6600, 'B-ORG': 6321, 'I-PER': 4528, 'I-ORG': 3704, 'B-MISC': 3438, 'I-LOC': 1157, 'I-MISC': 1155})


In [180]:
dev_sentences_raw = read_data("/home/omghag/CSCI-544-Assignment/HW3/data/dev")  # no DOCSTART filter

In [181]:
def write_predictions(sentences, preds, idx2tag, output_file, include_docstart=True):
    with open(output_file, "w") as f:
        flat_idx = 0
        for sentence in sentences:
            if include_docstart and sentence[0][0] == '-DOCSTART-':
                f.write("1 -DOCSTART- O\n\n")
                continue
            for i, (word, _) in enumerate(sentence):
                pred_tag = idx2tag[preds[flat_idx]]
                f.write(f"{i+1} {word} {pred_tag}\n")
                flat_idx += 1
            f.write("\n")

In [182]:
write_predictions(dev_sentences_raw, preds ,idx2tag, "dev1.out")

In [168]:
model.load_state_dict(torch.load('/home/omghag/CSCI-544-Assignment/HW3/blstm1.pt'))

<All keys matched successfully>